# Optuna Hyperparameter Tuning XGBoost (50 Trials)
Notebook ini fokus tuning model XGBoost untuk target `luas_km2`.

Alur:
1. Load data hasil encoding
2. Split train/val/test = 80/10/10
3. Tuning Optuna sebanyak 50 trial
4. Train model terbaik
5. Evaluasi test set
6. Simpan model dan artefak tuning

In [1]:
from pathlib import Path
from time import perf_counter
import importlib
import json
import pickle
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Install package jika belum ada
def ensure_package(module_name, pip_name=None):
    pip_name = pip_name or module_name
    try:
        return importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        return importlib.import_module(module_name)

optuna = ensure_package('optuna')
xgboost = ensure_package('xgboost')

from xgboost import XGBRegressor

print('Optuna version:', optuna.__version__)
print('XGBoost version:', xgboost.__version__)

Optuna version: 4.6.0
XGBoost version: 3.0.5


In [ ]:
# Konfigurasi
step_start = perf_counter()

RANDOM_STATE = 42
N_TRIALS = 50
TARGET_COL = 'luas_km2'

BASE_DIR = Path('.')
DATA_PATH = BASE_DIR / 'outlier_treatment_outputs/java-ash-hysplit_handled.csv'
ARTIFACT_DIR = BASE_DIR / 'artifacts_optuna_luas_xgboost'
MODEL_DIR = ARTIFACT_DIR / 'model'
METRIC_DIR = ARTIFACT_DIR / 'metrics'
TUNING_DIR = ARTIFACT_DIR / 'tuning'

for d in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, TUNING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

process_times = {}
process_times['setup'] = perf_counter() - step_start
print(f"Waktu setup: {process_times['setup']:.6f} detik")

Waktu setup: 0.002799 detik


In [4]:
# Load data
step_start = perf_counter()

df = pd.read_csv(DATA_PATH)
print('Shape data:', df.shape)

required_cols = [TARGET_COL]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f'Kolom wajib tidak ditemukan: {missing}')

target_multi_cols = ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']
feature_cols = [c for c in df.columns if c not in target_multi_cols]

X = df[feature_cols].copy()
y_raw = df[TARGET_COL].copy()

# Transform target untuk stabilitas training
y = np.log1p(y_raw)

process_times['load_data'] = perf_counter() - step_start
print(f"Waktu load_data: {process_times['load_data']:.6f} detik")
print('Jumlah fitur:', len(feature_cols))
df.head()

Shape data: (1707, 19)
Waktu load_data: 0.012704 detik
Jumlah fitur: 15


,timestamp,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km,volcano_filter_Bromo,volcano_filter_Merapi,volcano_filter_Raung,volcano_filter_Semeru,volcano_filter_Slamet
0,6,3,-7.54194,110.44194,2968,1000,4.7,270.0,50,30,32.511505,614.390650,303.257335,18.304149,0,1,0,0,0
1,1,1,-7.54194,110.44194,2968,1000,8.0,135.0,51,168,27.761756,460.792988,37.901873,15.892819,0,1,0,0,0
2,3,1,-7.54194,110.44194,2968,1200,7.1,90.0,65,133,44.625714,614.214150,265.635506,23.934437,0,1,0,0,0
3,3,1,-7.54194,110.44194,2968,1500,3.2,135.0,70,160,22.998759,184.285530,288.108280,11.770659,0,1,0,0,0
4,3,1,-7.54194,110.44194,2968,2500,8.3,315.0,70,145,39.159595,307.107075,270.462663,20.809694,0,1,0,0,0


In [5]:
# Split data: 80/10/10
step_start = perf_counter()

X_train, X_temp, y_train, y_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y, y_raw, test_size=0.2, random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp, y_temp_raw, test_size=0.5, random_state=RANDOM_STATE
)

process_times['split_data'] = perf_counter() - step_start
print(f"Waktu split_data: {process_times['split_data']:.6f} detik")
print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)

Waktu split_data: 0.007770 detik
Train: (1365, 15) | Val: (171, 15) | Test: (171, 15)


In [6]:
# Objective function Optuna
def objective(trial):
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'n_estimators': trial.suggest_int('n_estimators', 800, 4000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 50.0, log=True),
    }

    model = XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    pred_val_log = model.predict(X_val)
    pred_val = np.expm1(pred_val_log)
    pred_val = np.clip(pred_val, a_min=0.0, a_max=None)

    rmse = np.sqrt(mean_squared_error(y_val_raw, pred_val))
    return rmse

In [7]:
# Jalankan tuning Optuna
step_start = perf_counter()

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

process_times['optuna_tuning'] = perf_counter() - step_start
print(f"Waktu optuna_tuning: {process_times['optuna_tuning']:.6f} detik")
print('Best RMSE (val):', study.best_value)
print('Best params:', study.best_params)

[I 2026-03-14 17:51:37,491] A new study created in memory with name: no-name-56a9cad9-c311-4fe9-aa82-9d865feb9abf


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-14 17:51:38,806] Trial 0 finished with value: 276.5112666909652 and parameters: {'n_estimators': 1998, 'learning_rate': 0.044635901521768134, 'max_depth': 8, 'min_child_weight': 12, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.006763888939818983}. Best is trial 0 with value: 276.5112666909652.
[I 2026-03-14 17:51:40,878] Trial 1 finished with value: 281.8729357647273 and parameters: {'n_estimators': 3066, 'learning_rate': 0.005242693862597309, 'max_depth': 10, 'min_child_weight': 17, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.0012291273711520685}. Best is trial 0 with value: 276.5112666909652.
[I 2026-03-14 17:51:41,916] Trial 2 finished with value: 282.4348535553088 and parameters: {'n_estimators': 2182, 'learning_rate': 0.009776854331372627, 'max_depth': 7, 'mi

In [8]:
# Train final model dengan parameter terbaik
step_start = perf_counter()

best_params = {
    **study.best_params,
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'tree_method': 'hist',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
}

best_model = XGBRegressor(**best_params)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

pred_test_log = best_model.predict(X_test)
pred_test = np.expm1(pred_test_log)
pred_test = np.clip(pred_test, a_min=0.0, a_max=None)

test_mae = mean_absolute_error(y_test_raw, pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_test))
test_r2 = r2_score(y_test_raw, pred_test)

metrics = {
    'target': TARGET_COL,
    'test_MAE': float(test_mae),
    'test_RMSE': float(test_rmse),
    'test_R2': float(test_r2),
    'best_val_RMSE': float(study.best_value),
    'n_trials': int(N_TRIALS)
}

process_times['train_final_model'] = perf_counter() - step_start
print(f"Waktu train_final_model: {process_times['train_final_model']:.6f} detik")
print(metrics)

Waktu train_final_model: 1.540175 detik
{'target': 'luas_km2', 'test_MAE': 198.33432606420797, 'test_RMSE': 363.00531238890454, 'test_R2': 0.40249923301358836, 'best_val_RMSE': 246.71087012563444, 'n_trials': 50}


In [9]:
# Simpan artefak
step_start = perf_counter()

with open(MODEL_DIR / 'best_xgb_luas_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open(TUNING_DIR / 'best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=2)

with open(METRIC_DIR / 'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

trials_df = study.trials_dataframe()
trials_df.to_csv(TUNING_DIR / 'optuna_trials.csv', index=False)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)
importance_df.to_csv(METRIC_DIR / 'feature_importance.csv', index=False)

process_times['save_artifacts'] = perf_counter() - step_start
print(f"Waktu save_artifacts: {process_times['save_artifacts']:.6f} detik")
print('Artefak tersimpan di:', ARTIFACT_DIR)
display(importance_df.head(15))

Waktu save_artifacts: 0.082844 detik
Artefak tersimpan di: artifacts_optuna_luas_xgboost


,feature,importance
0,timestamp,0.144213
1,volcano_filter_Raung,0.128389
2,volcano_filter_Semeru,0.123608
3,elevation,0.094565
4,latitude,0.093777
5,longitude,0.091344
6,tinggi_letusan_m,0.082889
7,alert_level,0.055064
8,kec_angin_km_jam,0.052692
9,amplitudo,0.043513


In [10]:
# Ringkasan metrik dan waktu proses
metrics_df = pd.DataFrame([metrics])
timing_df = pd.DataFrame([
    {'process': k, 'duration_seconds': v} for k, v in process_times.items()
]).sort_values('duration_seconds', ascending=False).reset_index(drop=True)

timing_df.to_csv(METRIC_DIR / 'process_times.csv', index=False)

display(metrics_df)
display(timing_df)

print('Notebook tuning selesai. Jalankan semua cell dari atas ke bawah.')

,target,test_MAE,test_RMSE,test_R2,best_val_RMSE,n_trials
0,luas_km2,198.334326,363.005312,0.402499,246.71087,50


,process,duration_seconds
0,optuna_tuning,79.110367
1,train_final_model,1.540175
2,save_artifacts,0.082844
3,load_data,0.012704
4,split_data,0.007770
5,setup,0.002799


Notebook tuning selesai. Jalankan semua cell dari atas ke bawah.
